# Vetch Quick Start

**Planet-aware observability for LLM inference.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prismatic-labs/vetch/blob/main/demo.ipynb)

Vetch wraps your LLM API calls to track energy, carbon, and cost—without reading prompts or completions.

**v0.1.6 New Features:**
- Session Aggregation for agentic AI (CrewAI, LangGraph, AutoGPT)
- `instrument()` / `uninstrument()` for global auto-tracking
- Azure OpenAI support
- Cache-aware pricing (Anthropic & OpenAI prompt caching)
- `vetch status` CLI command
- Alert cooldown throttling
- Session memory safeguards (OOM prevention)

## Installation

In [ ]:
!pip install -q vetch

## CLI: Estimate Without Code

The fastest way to see Vetch in action—estimate energy and carbon for any model:

In [ ]:
!vetch estimate --model gpt-4o --input-tokens 1000 --output-tokens 500 --region us-east-1

## CLI: Compare Models

Compare energy, carbon, and cost across different models:

In [ ]:
!vetch compare --models gpt-4o,gpt-4o-mini,claude-3-5-sonnet --input-tokens 1000 --output-tokens 500

## CLI: Check Vetch Status

See your registry, grid API, and provider configuration at a glance:

In [ ]:
!vetch status

---

## NEW in v0.1.6: Global Auto-Instrumentation

One line at startup to track **all** LLM calls automatically—no `wrap()` needed:

In [ ]:
import vetch

# One line at startup — all LLM calls are tracked automatically
# vetch.instrument(region="us-east-1", tags={"service": "chat-api"})

# Works with OpenAI, Anthropic, Azure OpenAI, and Vertex AI:
# client = openai.OpenAI()
# response = client.chat.completions.create(model="gpt-4o", ...)
# # Energy, cost, and carbon events emitted automatically!

# Clean teardown for tests:
# vetch.uninstrument()  # Restore original SDK methods

print("instrument() supported providers:")
print("  - OpenAI (openai.OpenAI, openai.AsyncOpenAI)")
print("  - Anthropic (anthropic.Anthropic, anthropic.AsyncAnthropic)")
print("  - Azure OpenAI (openai.AzureOpenAI, openai.AsyncAzureOpenAI)")
print("  - Vertex AI (vertexai.generative_models.GenerativeModel)")
print("  - OpenAI-compatible (OpenRouter, Together.ai, Ollama, vLLM)")
print()
print("Safe to call multiple times (idempotent).")
print("Set VETCH_DISABLED=true as emergency kill switch.")

---

## NEW in v0.1.6: Session Aggregation (Agentic AI)

Group multiple LLM calls into sessions for agentic frameworks (CrewAI, AutoGPT, LangGraph).
Sessions track cumulative energy, cost, and carbon with built-in OOM safeguards.

In [ ]:
import vetch

# Sessions aggregate metrics across multiple LLM calls
with vetch.Session(tags={"agent": "researcher", "task": "summarize"}, emit=False) as session:
    # Each wrap() call inside registers with the session automatically
    # with vetch.wrap() as ctx1:
    #     response1 = client.chat.completions.create(...)
    # with vetch.wrap() as ctx2:
    #     response2 = client.chat.completions.create(...)

    # Simulate events for demo
    session.register_event({
        "model": "gpt-4o", "provider": "openai",
        "estimated_energy_wh": 0.05, "estimated_carbon_g": 2.5,
        "estimated_cost_usd": 0.03, "cache_read_tokens": 500,
        "usage": {"text": {"input_tokens": 1000, "output_tokens": 200}},
    })
    session.register_event({
        "model": "claude-3-5-sonnet", "provider": "anthropic",
        "estimated_energy_wh": 0.08, "estimated_carbon_g": 4.0,
        "estimated_cost_usd": 0.05, "cache_read_tokens": 800,
        "usage": {"text": {"input_tokens": 2000, "output_tokens": 500}},
    })

print(f"Session ID: {session.session_id}")
print(f"Call count: {session.call_count}")
print(f"Total energy: {session.total_energy_wh:.4f} Wh")
print(f"Total carbon: {session.total_carbon_g:.4f} gCO2e")
print(f"Total cost: ${session.total_cost_usd:.4f}")
print(f"Total cache reads: {session.total_cache_read_tokens} tokens")
print(f"Models used: {session.models_used}")
print(f"Providers used: {session.providers_used}")
print(f"Duration: {session.duration_ms:.1f} ms")

# Safety: max_calls prevents OOM in long-running loops (default: 10,000)
# vetch.Session(max_calls=1000)  # Custom limit

---

## Budget Alerts

Set cost/energy/carbon thresholds. **Alerts are warn-only—they never block your inference.**

New in v0.1.6: `alert_cooldown_seconds` prevents alert flooding in runaway loops (default: 60s).

In [ ]:
import vetch

# Set a budget: warn when cost > $0.01 per request
vetch.set_budget("per-request", cost_usd=0.01, warn_at_pct=80)

# Set a session budget: warn when total energy > 0.1 Wh
vetch.set_budget("session-energy", energy_wh=0.1, window="session")

# Check current budget status
print("Configured budgets:")
print(vetch.get_budget_status())

In [ ]:
# Register a callback for budget alerts
@vetch.on_budget_alert
def my_alert_handler(alert):
    print(f"🚨 BUDGET ALERT: {alert}")

print("Alert callback registered!")
print("When you run inference, alerts will fire if thresholds are approached.")

### Budget Alerts via Environment Variables

You can also configure budgets without code:

In [ ]:
import os

# These environment variables auto-configure budgets on import:
# os.environ["VETCH_BUDGET_COST_USD"] = "0.10"       # Per-request cost limit
# os.environ["VETCH_BUDGET_ENERGY_WH"] = "0.001"     # Per-request energy limit
# os.environ["VETCH_BUDGET_SESSION_COST_USD"] = "1.00"  # Session-wide limit

print("Environment variables for budget alerts:")
print("  VETCH_BUDGET_COST_USD        - Per-request cost threshold")
print("  VETCH_BUDGET_ENERGY_WH       - Per-request energy threshold")
print("  VETCH_BUDGET_CARBON_G        - Per-request carbon threshold")
print("  VETCH_BUDGET_SESSION_COST_USD - Session-wide cost threshold")

---

## OTLP Export

Export Vetch metrics to any OTLP-compatible backend (Datadog, Honeycomb, Grafana, Jaeger).

In [ ]:
# Configure OTLP export (requires opentelemetry-sdk)
# !pip install -q opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc

# from vetch import configure_otlp_export

# Example: Export to Honeycomb
# configure_otlp_export(
#     endpoint="https://api.honeycomb.io",
#     headers={"x-honeycomb-team": "your-api-key"}
# )

# Example: Export to Grafana Cloud
# configure_otlp_export(
#     endpoint="https://otlp-gateway-prod-us-central-0.grafana.net/otlp",
#     headers={"Authorization": "Basic base64-credentials"}
# )

# Example: Export to local Jaeger
# configure_otlp_export(endpoint="http://localhost:4317")

print("OTLP Export Options:")
print("  - Datadog")
print("  - Honeycomb")
print("  - Grafana Cloud")
print("  - Jaeger")
print("  - Any OTLP-compatible backend")
print("")
print("Metrics exported:")
print("  - vetch.energy_wh (histogram)")
print("  - vetch.carbon_g (histogram)")
print("  - vetch.cost_usd (histogram)")
print("  - vetch.requests_total (counter)")

### Auto-Configure via Environment

In [ ]:
import os

# Auto-configure OTLP export from environment:
# os.environ["VETCH_OTEL_EXPORT"] = "true"
# os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://api.honeycomb.io"
# os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = "x-honeycomb-team=your-key"

print("Environment variables for OTLP:")
print("  VETCH_OTEL_EXPORT=true           - Enable automatic export")
print("  OTEL_EXPORTER_OTLP_ENDPOINT      - OTLP endpoint URL")
print("  OTEL_EXPORTER_OTLP_HEADERS       - Auth headers (key=value,key2=value2)")
print("  VETCH_OTEL_SERVICE_NAME          - Service name (default: vetch)")

---

## Cache-Aware Pricing

Vetch detects prompt cache hits (Anthropic & OpenAI) and calculates accurate costs.

New in v0.1.6: Cache discounts are applied automatically in cost calculations.

In [ ]:
# When using prompt caching, Vetch captures:
# - cache_read_tokens: Tokens served from cache (cheaper!)
# - cache_creation_tokens: Tokens written to cache
# - cache_hit: Boolean indicating any cache usage

# Example event fields:
example_event = {
    "cache_read_tokens": 500,      # These tokens were cached
    "cache_creation_tokens": 0,    # No new cache entries
    "cache_hit": True,             # Cache was used
    "estimated_cost_usd": 0.0015,  # Lower cost due to cache!
}

print("Cache Detection Fields:")
for k, v in example_event.items():
    print(f"  {k}: {v}")

print("\nNote: Anthropic charges 90% less for cached input tokens.")
print("OpenAI charges 50% less for cached tokens.")

---

## SDK Usage: wrap()

In real usage, you wrap your LLM calls:

In [ ]:
import os
os.environ["VETCH_OUTPUT"] = "none"  # Suppress JSON logging for cleaner output

from vetch import wrap

# In production, you'd make a real API call inside the context:
#
# from openai import OpenAI
# client = OpenAI()
#
# with wrap(region="us-east-1", tags={"team": "ml"}) as ctx:
#     response = client.chat.completions.create(
#         model="gpt-4o",
#         messages=[{"role": "user", "content": "Hello"}]
#     )
#
# print(f"Energy: {ctx.event['estimated_energy_wh']:.4f} Wh")
# print(f"Carbon: {ctx.event['estimated_carbon_g']:.4f} gCO2e")
# print(f"Cost: ${ctx.event['estimated_cost_usd']:.4f}")
# print(f"Cache Hit: {ctx.event.get('cache_hit', False)}")
# print(f"Budget Exceeded: {ctx.event.get('budget_exceeded', False)}")

print("wrap() usage example (no API key needed for CLI demos above)")

---

## Try It With Your API Key (Optional)

If you have an OpenAI or Anthropic API key, uncomment and run one of these:

In [ ]:
# Uncomment ONE section and add your credentials:

# --- OpenAI ---
# !pip install -q openai
# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."  # Add your key
# os.environ["VETCH_OUTPUT"] = "none"
#
# import vetch
# from openai import OpenAI
#
# # Set a budget alert
# vetch.set_budget("demo", cost_usd=0.01, warn_at_pct=50)
#
# client = OpenAI()
# with vetch.wrap(region="us-east-1", tags={"demo": "colab"}) as ctx:
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": "Say hello in 5 words"}]
#     )
#     print(response.choices[0].message.content)
#
# print(f"\n--- Vetch Metrics ---")
# print(f"Energy: {ctx.event['estimated_energy_wh']:.6f} Wh")
# print(f"Carbon: {ctx.event['estimated_carbon_g']:.6f} gCO2e")
# print(f"Cost:   ${ctx.event['estimated_cost_usd']:.6f}")
# print(f"Cache Hit: {ctx.event.get('cache_hit', False)}")
# print(f"Budget Exceeded: {ctx.event.get('budget_exceeded', False)}")

# --- Anthropic ---
# !pip install -q anthropic
# import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # Add your key
# os.environ["VETCH_OUTPUT"] = "none"
#
# import vetch
# import anthropic
#
# # Set a budget alert
# vetch.set_budget("demo", cost_usd=0.01, warn_at_pct=50)
#
# client = anthropic.Anthropic()
# with vetch.wrap(region="us-west-2", tags={"demo": "colab"}) as ctx:
#     response = client.messages.create(
#         model="claude-3-5-haiku-latest",
#         max_tokens=50,
#         messages=[{"role": "user", "content": "Say hello in 5 words"}]
#     )
#     print(response.content[0].text)
#
# print(f"\n--- Vetch Metrics ---")
# print(f"Energy: {ctx.event['estimated_energy_wh']:.6f} Wh")
# print(f"Carbon: {ctx.event['estimated_carbon_g']:.6f} gCO2e")
# print(f"Cost:   ${ctx.event['estimated_cost_usd']:.6f}")
# print(f"Cache Read Tokens: {ctx.event.get('cache_read_tokens', 'N/A')}")
# print(f"Budget Exceeded: {ctx.event.get('budget_exceeded', False)}")

# --- Vertex AI (Google Cloud) ---
# !pip install -q google-cloud-aiplatform
# # Authenticate: Run `gcloud auth application-default login` or set GOOGLE_APPLICATION_CREDENTIALS
#
# import vertexai
# from vertexai.generative_models import GenerativeModel
# from vetch import wrap
#
# vertexai.init(project="your-project-id", location="us-central1")
# model = GenerativeModel("gemini-1.5-flash")
#
# with wrap(region="us-central1", tags={"demo": "colab"}) as ctx:
#     response = model.generate_content("Say hello in 5 words")
#     print(response.text)
#
# print(f"\n--- Vetch Metrics ---")
# print(f"Energy: {ctx.event['estimated_energy_wh']:.6f} Wh")
# print(f"Carbon: {ctx.event['estimated_carbon_g']:.6f} gCO2e")
# print(f"Cost:   ${ctx.event['estimated_cost_usd']:.6f}")

print("Uncomment a section above and add your credentials to try it live!")

---

## Session Statistics & Advisories

Track patterns across multiple requests and get optimization recommendations:

In [ ]:
from vetch import get_session_stats, generate_advisories

# After making multiple calls, check session stats
stats = get_session_stats()
print("Session Stats:")
print(f"  Total Requests: {stats.total_requests}")
print(f"  Total Energy: {stats.total_energy_wh:.4f} Wh")
print(f"  Total Carbon: {stats.total_carbon_g:.4f} gCO2e")
print(f"  Total Cost: ${stats.total_cost_usd:.4f}")

# Get optimization advisories
advisories = generate_advisories(stats)
if advisories:
    print("\nAdvisories:")
    for a in advisories:
        print(f"  [{a.level.value}] {a.title}")
        print(f"    {a.description}")
else:
    print("\nNo advisories yet (make some API calls first!)")

---

## View Methodology

Understand how energy estimates are calculated:

In [ ]:
!vetch methodology | head -80

---

## Energy Tiers Explained

Vetch uses a tiered system for estimate confidence:

In [ ]:
tiers = [
    (0, "Measured", "±10-20%", "Direct GPU measurement (pynvml)"),
    (1, "Vendor-Published", "±20-50%", "Official provider data"),
    (2, "Validated", "±50-100%", "Crowdsourced aggregates"),
    (3, "Estimated", "order of magnitude", "Parameter-based calculation"),
]

print("Energy Estimate Tiers:")
print("-" * 70)
print(f"{'Tier':<6} {'Name':<18} {'Uncertainty':<18} {'Source'}")
print("-" * 70)
for tier, name, uncertainty, source in tiers:
    print(f"{tier:<6} {name:<18} {uncertainty:<18} {source}")

print("\nNote: Most cloud models are Tier 3 (estimated) in alpha.")
print("Use GPU calibration for Tier 0 measurements on local inference.")

---

## Grid Carbon Intensity

Vetch uses a 4-tier fallback for grid data:

In [ ]:
fallback_tiers = [
    (1, "Memory Cache", "5 min TTL", "Fastest, per-process"),
    (2, "File Cache", "~/.vetch/", "Shared across processes"),
    (3, "Electricity Maps API", "Live data", "Requires ELECTRICITY_MAPS_API_KEY"),
    (4, "Regional Averages", "Static", "Built-in fallback (~24h stale)"),
]

print("Grid Intensity Fallback Hierarchy:")
print("-" * 70)
print(f"{'Tier':<6} {'Source':<22} {'TTL/Location':<18} {'Notes'}")
print("-" * 70)
for tier, source, ttl, notes in fallback_tiers:
    print(f"{tier:<6} {source:<22} {ttl:<18} {notes}")

print("\nFor live grid data, get a free API key from electricitymaps.com")
print("and set: ELECTRICITY_MAPS_API_KEY=your-key")

---

## Feedback

This is an **alpha release**. We'd love your feedback!

- **Issues**: [github.com/prismatic-labs/vetch/issues](https://github.com/prismatic-labs/vetch/issues)
- **Docs**: [github.com/prismatic-labs/vetch](https://github.com/prismatic-labs/vetch)

### Questions to consider:
- Are the CLI commands useful for your workflow?
- Is the budget alert system helpful?
- What observability backends would you like to see?
- What providers/models are you missing?
- What would make this more valuable?